In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from static_token_div.tools.tools import sigmoid
from static_token_div.tools.w2v_tools import _create_learning_file
import warnings
warnings.filterwarnings('ignore')

In [2]:
text_path = "../../resources/tlnl_tp1_data/alexandre_dumas/Le_comte_de_Monte_Cristo.tok"
word_except = ["<s>", "</s>"]

data = _create_learning_file(
    text_path=text_path,
    word_except=word_except,
    k=10,
    L=2,
    minc=5
)

In [9]:
data = np.array(data)
data

array([[   1,    3, 4175, ..., 9438, 3251, 4114],
       [   3,    1, 3437, ..., 2728, 1887, 6786],
       [   3,    4, 1983, ...,   55, 1393, 4639],
       ...,
       [9828,   22, 2346, ...,  602, 3301, 1754],
       [  22,   32, 6213, ..., 1375,  828, 1864],
       [  22, 9828,  957, ..., 2977, 1313,  149]])

# Extract data

In [21]:
target = data[:, 0]
c_pos = data[:, 1]
c_neg = data[:, 2:]
target.shape, c_pos.shape, c_neg.shape

((669842,), (669842,), (669842, 10))

In [22]:
vocab_size = data.shape[0]
vocab_size

3154

# Process

In [16]:
def safe_log(x):
    epsilon = 1e-10
    return np.log(np.clip(x, epsilon, 1.0))

def loss_function(m, c_pos, c_neg):
    pos_loss = safe_log(sigmoid(np.dot(m, c_pos)))
    neg_loss =  np.sum(safe_log(sigmoid(-np.dot(m, c_neg.T))))
    return - (pos_loss + neg_loss)

In [17]:
embedding_dim = 100
W = np.random.rand(np.unique(data).shape[0] + 1, embedding_dim)
C = np.random.rand(np.unique(data).shape[0] + 1, embedding_dim)
learning_rate = 0.1
iterations = 5
losses = []

In [18]:
def train(W, C):
    for _ in range(iterations):
        losses = []

        for line in data:
            m_idx = line[0]
            cpos_idx = line[1]
            cneg_idx = line[2:]

            m = W[m_idx]
            cpos = C[cpos_idx]
            cneg = np.array([C[i] for i in cneg_idx])

            losses.append(loss_function(m, cpos, cneg))

            grad_pos = (sigmoid(np.dot(m, cpos)) - 1) * m
            C[cpos_idx] -= learning_rate * grad_pos

            for idx in cneg_idx:
                cneg_vec = C[idx]
                grad_neg = sigmoid(np.dot(m, cneg_vec)) * m
                C[idx] -= learning_rate * grad_neg


            grad_target_pos = (sigmoid(np.dot(m, cpos)) - 1) * cpos
            grad_target_neg = np.sum([sigmoid(np.dot(m, C[i])) * C[i] for i in cneg_idx], axis=0)
            W[m_idx] -= learning_rate * (grad_target_pos + grad_target_neg)


        print(np.mean(losses))

train(W, C)

IndexError: index 4175 is out of bounds for axis 0 with size 3155